# BigQuery: Global Queries Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/POH/blob/main/bq_global_queries_demo.ipynb)

This notebook demonstrates the **Global Queries** feature in BigQuery, which allows you to reference data stored in more than one region in a single query without needing to physically move or copy the data.

## Use Case
A global enterprise has sales data in `us-central1` and inventory data in `europe-west1`. Previously, they had to export/import data to join these tables. Now, they can join them directly.

### Requirements
- BigQuery API enabled.
- **Permissions**: `bigquery.jobs.createGlobalQuery` (included in `BigQuery Admin`).
- Datasets in at least two different regions (e.g., `us-central1` and `europe-west1`).

In [ ]:
# 1. Setup and Authentication
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

from google.cloud import bigquery
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
client = bigquery.Client(project=project_id)

### 2. [MANDATORY] Enable Global Queries for your Project

By default, cross-region joins are disabled to prevent accidental data transfer costs. You must run these DDL statements once to opt-in.

In [ ]:
# Enable Execution in the US (the primary region for this demo)
setup_query_exec = f"""
ALTER PROJECT SET OPTIONS (
  `region-us.enable_global_queries_execution` = true
);
"""

# Enable Data Access in EU-West1 (the remote region for this demo)
setup_query_access = f"""
ALTER PROJECT SET OPTIONS (
  `region-europe-west1.enable_global_queries_data_access` = true
);
"""

print("Enabling Global Query features...")
try:
    client.query(setup_query_exec).result()
    client.query(setup_query_access).result()
    print("Success: Project configured for Global Queries.")
except Exception as e:
    print(f"Warning: Could not configure project. Ensure you have 'BigQuery Admin' role. Error: {str(e)}")

### 3. [PREREQUISITES] Create Datasets, Tables, and Dummy Data

This block sets up the environment across two regions: `us-central1` and `europe-west1`.

In [ ]:
def setup_demo_data():
    us_dataset_id = f"{project_id}.us_sales"
    eu_dataset_id = f"{project_id}.eu_inventory"
    
    # Create US Dataset (us-central1)
    us_dataset = bigquery.Dataset(us_dataset_id)
    us_dataset.location = "us-central1"
    client.create_dataset(us_dataset, exists_ok=True)
    
    # Create EU Dataset (europe-west1)
    eu_dataset = bigquery.Dataset(eu_dataset_id)
    eu_dataset.location = "europe-west1"
    client.create_dataset(eu_dataset, exists_ok=True)

    # Create US Sales Table
    sales_table_id = f"{us_dataset_id}.sales_data"
    sales_schema = [
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("units_sold", "INTEGER"),
        bigquery.SchemaField("sale_date", "DATE"),
    ]
    client.create_table(bigquery.Table(sales_table_id, schema=sales_schema), exists_ok=True)
    client.insert_rows_json(sales_table_id, [
        {"product_id": "PROD_998", "units_sold": 450, "sale_date": "2026-03-09"},
        {"product_id": "PROD_442", "units_sold": 120, "sale_date": "2026-03-09"}
    ])

    # Create EU Inventory Table
    inventory_table_id = f"{eu_dataset_id}.inventory_data"
    inventory_schema = [
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("stock_level", "INTEGER"),
        bigquery.SchemaField("warehouse_location", "STRING"),
    ]
    client.create_table(bigquery.Table(inventory_table_id, schema=inventory_schema), exists_ok=True)
    client.insert_rows_json(inventory_table_id, [
        {"product_id": "PROD_998", "stock_level": 1200, "warehouse_location": "Frankfurt-DE"},
        {"product_id": "PROD_442", "stock_level": 300, "warehouse_location": "London-UK"}
    ])
    print("Demo data prepared successfully.")

setup_demo_data()

### 4. Define and Execute the Global Query

We specify `location='US'` to tell BigQuery to use the US region as the compute primary.

In [ ]:
query = f"""
SELECT
    sales.product_id,
    sales.units_sold,
    inventory.stock_level,
    inventory.warehouse_location
FROM
    `{project_id}.us_sales.sales_data` AS sales
INNER JOIN
    `{project_id}.eu_inventory.inventory_data` AS inventory
ON
    sales.product_id = inventory.product_id
WHERE
    sales.sale_date = '2026-03-09'
"""

print("Executing Global Query (Primary: US)...")
job_config = bigquery.QueryJobConfig(use_legacy_sql=False)
query_job = client.query(query, location="US", job_config=job_config)
results = query_job.to_dataframe()
results.head()

### 5. Things to remember or know
- **Zero Data Movement**: No more costly and slow ETL pipelines to sync data across regions just for reporting.
- **One-Time Setup**: The `ALTER PROJECT` opt-in is only required once per project/region pair.
- **Preview Pricing**: Data transfer between regions is billed at standard replication rates during processing.